In [4]:
"""
Audit Logs - Bronze to Silver Layer
Parses JSON and extracts key fields for analytics
Uses incremental processing with state tracking

UPDATED: Now writes to Delta table instead of Parquet volume
"""

from pyspark.sql import functions as F

# ============================================================================
# CONFIGURATION
# ============================================================================

BRONZE_PATH = "/Volumes/gitrepo/default/git_oci_aidp_bronze/audit_logs/data"

# NEW: Delta table instead of Parquet path
CATALOG = "gitrepo"
SCHEMA = "default"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_audit_logs"

# State tracking (unchanged)
STATE_PATH = "/Volumes/gitrepo/default/git_oci_aidp_silver/audit_logs/state"

# Spark tuning
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("=" * 70)
print("AUDIT LOGS - BRONZE TO SILVER (DELTA TABLE)")
print("=" * 70)
print(f"Bronze Path: {BRONZE_PATH}")
print(f"Silver Table: {SILVER_TABLE}")
print(f"State Path: {STATE_PATH}")
print("=" * 70)

# ============================================================================
# LOAD LAST PROCESSED TIMESTAMP
# ============================================================================

print("\n[STEP 1] Loading checkpoint state...")

last_ts = None
try:
    state_df = spark.read.parquet(STATE_PATH)
    row = state_df.select("last_ingest_time").orderBy(F.col("last_ingest_time").desc()).first()
    last_ts = row[0] if row else None
except Exception:
    last_ts = None

print(f"Last processed ingest_time: {last_ts}")

# ============================================================================
# READ BRONZE (Incremental)
# ============================================================================

print("\n[STEP 2] Reading bronze layer...")

bronze_df = spark.read.parquet(BRONZE_PATH)

# Incremental filter
if last_ts is not None:
    bronze_df = bronze_df.filter(F.col("ingest_time") > last_ts)

# Check if there's new data
if bronze_df.limit(1).count() == 0:
    print("No new bronze rows. Silver is up to date.")
else:
    print(f"Found new data to process")
    
    # ============================================================================
    # PARSE JSON - Extract Key Fields
    # ============================================================================
    
    print("\n[STEP 3] Parsing JSON and extracting fields...")
    
    silver_df = (
        bronze_df
        .withColumn("ingest_date", F.to_date("ingest_time"))
        
        # ---- Top-level fields ----
        .withColumn("event_time_str", F.get_json_object("raw_json", "$.time"))
        .withColumn("event_time", F.to_timestamp("event_time_str"))
        .withColumn("event_type", F.get_json_object("raw_json", "$.type"))
        .withColumn("event_source", F.get_json_object("raw_json", "$.source"))
        
        # ---- $.data.* fields ----
        .withColumn("event_name", F.get_json_object("raw_json", "$.data.eventName"))
        .withColumn("event_id", F.get_json_object("raw_json", "$.data.eventId"))
        
        # Compartment info
        .withColumn("compartment_id", F.get_json_object("raw_json", "$.data.compartmentId"))
        .withColumn("compartment_name", F.get_json_object("raw_json", "$.data.compartmentName"))
        .withColumn("tenancy_id", F.get_json_object("raw_json", "$.data.tenantId"))
        
        # Resource info
        .withColumn("resource_name", F.get_json_object("raw_json", "$.data.resourceName"))
        .withColumn("resource_id", F.get_json_object("raw_json", "$.data.resourceId"))
        .withColumn("availability_domain", F.get_json_object("raw_json", "$.data.availabilityDomain"))
        
        # ---- $.data.identity.* fields ----
        .withColumn("principal_id", F.get_json_object("raw_json", "$.data.identity.principalId"))
        .withColumn("principal_name", F.get_json_object("raw_json", "$.data.identity.principalName"))
        .withColumn("auth_type", F.get_json_object("raw_json", "$.data.identity.authType"))
        .withColumn("caller_name", F.get_json_object("raw_json", "$.data.identity.callerName"))
        .withColumn("user_agent", F.get_json_object("raw_json", "$.data.identity.userAgent"))
        .withColumn("ip_address", F.get_json_object("raw_json", "$.data.identity.ipAddress"))
        .withColumn("credentials", F.get_json_object("raw_json", "$.data.identity.credentials"))
        
        # ---- $.data.request.* fields ----
        .withColumn("request_action", F.get_json_object("raw_json", "$.data.request.action"))
        .withColumn("request_headers", F.get_json_object("raw_json", "$.data.request.headers"))
        .withColumn("request_id", F.get_json_object("raw_json", "$.data.request.id"))
        .withColumn("request_path", F.get_json_object("raw_json", "$.data.request.path"))
        .withColumn("request_parameters", F.get_json_object("raw_json", "$.data.request.parameters"))
        
        # ---- $.data.response.* fields ----
        .withColumn("response_status", F.get_json_object("raw_json", "$.data.response.status"))
        .withColumn("response_time", F.get_json_object("raw_json", "$.data.response.responseTime"))
        .withColumn("response_message", F.get_json_object("raw_json", "$.data.response.message"))
        .withColumn("response_payload", F.get_json_object("raw_json", "$.data.response.payload"))
        
        # ---- $.data.stateChange.* fields ----
        .withColumn("state_previous", F.get_json_object("raw_json", "$.data.stateChange.previous"))
        .withColumn("state_current", F.get_json_object("raw_json", "$.data.stateChange.current"))
        
        # ---- Additional details ----
        .withColumn("additional_details", F.get_json_object("raw_json", "$.data.additionalDetails"))
    )
    
    # Keep only rows with valid event data
    silver_df = silver_df.filter(
        F.col("event_name").isNotNull() | F.col("event_type").isNotNull()
    )
    
    print(f"Parsed fields from JSON")
    
    # ============================================================================
    # WRITE TO DELTA TABLE (instead of Parquet)
    # ============================================================================
    
    print("\n[STEP 4] Writing to Delta table...")
    
    (
        silver_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE)
    )
    
    print(f"✓ Data appended to {SILVER_TABLE}")
    
    # ============================================================================
    # UPDATE STATE
    # ============================================================================
    
    print("\n[STEP 5] Updating state...")
    
    new_last_ts = silver_df.select(F.max("ingest_time").alias("last_ingest_time")).first()[0]
    if new_last_ts is not None:
        (
            spark.createDataFrame([(new_last_ts,)], ["last_ingest_time"])
            .write.mode("overwrite")
            .parquet(STATE_PATH)
        )
        print(f"State updated to: {new_last_ts}")
    
    # ============================================================================
    # VERIFICATION
    # ============================================================================
    
    print("\n[STEP 6] Verifying silver Delta table...")
    
    silver_out_df = spark.table(SILVER_TABLE)
    
    total_records = silver_out_df.count()
    unique_events = silver_out_df.select("event_name").distinct().count()
    unique_principals = silver_out_df.select("principal_id").filter(F.col("principal_id").isNotNull()).distinct().count()
    
    print(f"\nSilver Layer Statistics:")
    print(f"  Total records: {total_records:,}")
    print(f"  Unique event types: {unique_events:,}")
    print(f"  Unique principals: {unique_principals:,}")
    
    # Show top events
    print("\nTop 10 events by frequency:")
    (silver_out_df
     .groupBy("event_name")
     .count()
     .orderBy(F.col("count").desc())
     .show(10, truncate=False))
    
    # Show sample parsed data
    print("\nSample parsed records:")
    (silver_out_df
     .select(
         "event_time",
         "event_name", 
         "principal_name",
         "compartment_name",
         "ip_address",
         "response_status"
     )
     .filter(F.col("event_name").isNotNull())
     .show(10, truncate=80))
    
    print("\n" + "=" * 70)
    print("✓ SILVER LAYER PROCESSING COMPLETED SUCCESSFULLY")
    print("=" * 70)
    print("\nNext Steps:")
    print("1. Run this script again to process new bronze records")
    print(f"2. Query silver layer: SELECT * FROM {SILVER_TABLE}")
    print("3. Move to Gold layer for aggregations and anomaly detection")
    print("=" * 70)

AUDIT LOGS - BRONZE TO SILVER (DELTA TABLE)
Bronze Path: /Volumes/gitrepo/default/git_oci_aidp_bronze/audit_logs/data
Silver Table: gitrepo.default.silver_audit_logs
State Path: /Volumes/gitrepo/default/git_oci_aidp_silver/audit_logs/state

[STEP 1] Loading checkpoint state...


Last processed ingest_time: 2026-01-28 00:41:50.385000

[STEP 2] Reading bronze layer...


No new bronze rows. Silver is up to date.


In [5]:
%sql
SELECT 
    MAX(ingest_date) as max_ingest_date,
    MAX(event_time) as max_event_time,
    COUNT(*) as total_records
FROM gitrepo.default.silver_audit_logs